<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Machine learning is iterative

**[Machine learning is iterative](https://learning.nextia-ai.com/courses/ml/m05/machine-learning-is-iterative/)** · Machine Learning: From Problem to Reliable Model · Module 5, lesson 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** compare experiments fairly (the same folds, the same metric, one change at a time) and keep an experiment log, the record of every try and its decision.

| | |
|---|---|
| **Time** | About 50 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | Cross-validation, from [Train, validation and test sets](https://learning.nextia-ai.com/courses/ml/m03/train-validation-and-test-sets/#cross-validation). Average precision, from [The precision–recall trade-off](https://learning.nextia-ai.com/courses/ml/m03/precision-recall-trade-off/#average-precision). The pipeline, from [Preprocess with a pipeline](https://learning.nextia-ai.com/courses/ml/m04/preprocess-with-a-pipeline/). |
| **You do not need** | The local project. The setup below downloads the data and writes `ticket_model.py`. |
| **Tested** | Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m05/machine-learning-is-iterative/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M05-L01-machine-learning-is-iterative/machine-learning-is-iterative-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the practice data of this course (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = []
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

## Setup: ticket_model.py

The next cell writes `ticket_model.py` as you wrote it in [Preprocess with a pipeline](https://learning.nextia-ai.com/courses/ml/m04/preprocess-with-a-pipeline/#write-ticket_modelpy): the column lists, `load()` and `build_pipeline()`. Run it. You should see `Writing ticket_model.py` (or `Overwriting`).

In [ ]:
%%writefile ticket_model.py
"""The shared parts of Larkfield's escalation model: the columns, how to load
a part of the data, and the preprocessing-plus-model pipeline.

Nextia Learning, C05, Module 4, lesson 3 (Preprocess with a pipeline).
"""

from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA = Path("data")
CATEGORIES = ["channel", "team", "segment", "region"]
NUMBERS = ["priority", "order_value", "word_count", "customer_tenure_days", "prior_tickets_90d",
           "created_hour"]
FEATURES = CATEGORIES + NUMBERS
TARGET = "escalated_72h"


def read(path):
    """Read a ticket CSV file. Only an empty cell is missing: the text
    "unknown" stays a category."""
    return pd.read_csv(path, parse_dates=["created_at"], dtype={"customer_id": "string"},
                       keep_default_na=False, na_values=[""])


def load(part):
    """One part of the split: train, valid or test."""
    return read(DATA / f"{part}.csv")


def build_pipeline(model=None):
    """Preprocessing and model in one object. fit() learns the medians, the
    scaling and the categories from the rows it is given, and nothing else."""
    numbers = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ])
    prepare = ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore"), CATEGORIES),
        ("numbers", numbers, NUMBERS),
    ])
    return Pipeline([("prepare", prepare), ("model", model if model is not None else LogisticRegression(max_iter=1000))])

## Setup: the experiment log

The next cell defines two helpers for your **experiment log**, the file `experiment_log.csv` in `ticket-model/`. You start the log in [Machine learning is iterative](https://learning.nextia-ai.com/courses/ml/m05/machine-learning-is-iterative/#the-experiment-log).

- `log_experiment(id, change, cv_ap, cv_std, valid_ap, decision, reason)` adds one row, with today's date. If the id is already in the log, it replaces that row, so you can run a cell again.
- `show_log()` shows the log as a table.

Run the cell. It prints nothing.

In [ ]:
# The experiment log: one row per experiment, in experiment_log.csv.
import datetime
from pathlib import Path

LOG_FILE = Path("experiment_log.csv")
LOG_COLUMNS = ["id", "date", "change", "cv_ap", "cv_std", "valid_ap", "decision", "reason"]


def log_experiment(id, change, cv_ap, cv_std, valid_ap=None, decision="", reason=""):
    """Add one row to the log (or replace the row with the same id)."""
    log = pd.read_csv(LOG_FILE, dtype={"date": str}) if LOG_FILE.exists() else pd.DataFrame(columns=LOG_COLUMNS)
    row = {"id": id, "date": datetime.date.today().isoformat(), "change": change,
           "cv_ap": round(float(cv_ap), 3), "cv_std": round(float(cv_std), 3),
           "valid_ap": None if valid_ap is None else round(float(valid_ap), 3),
           "decision": decision, "reason": reason}
    log = pd.concat([log[log["id"] != id], pd.DataFrame([row])], ignore_index=True)
    log.sort_values("id").to_csv(LOG_FILE, index=False)


def show_log(columns=("id", "change", "cv_ap", "cv_std", "valid_ap", "decision")):
    """The log as a table (without the date and the reason, unless you ask for them)."""
    log = pd.read_csv(LOG_FILE, dtype={"date": str}).sort_values("id")
    with pd.option_context("display.max_colwidth", 60, "display.width", 200):
        print(log[list(columns)].fillna("").to_string(index=False))

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Why you try several models

Nobody can say in advance which algorithm works best on a dataset. Each algorithm assumes a shape of pattern, and the data decides which shape fits. So practitioners **try several, measure each one in the same way, and keep what helps**. The work goes in rounds: this is why machine learning is called **iterative**. The [lesson page](https://learning.nextia-ai.com/courses/ml/m05/machine-learning-is-iterative/#the-loop) shows the loop as a diagram.

Every round must be measured in the same way: the same 5 folds of train, the same metric (average precision, AP), one change at a time.

Run the next cell. It loads the data and defines two helpers. `cv_scores()` gives the AP of a pipeline on each of the 5 folds. `rule_scores()` gives the AP of a fixed rule, which learns nothing, on the same 5 folds.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold, cross_val_score

from ticket_model import FEATURES, TARGET, build_pipeline, load

train, valid = load("train"), load("valid")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)


def cv_scores(pipeline):
    """The AP of a pipeline on each of the 5 folds of train (cross-validation)."""
    return cross_val_score(pipeline, train[FEATURES], train[TARGET], cv=cv, scoring="average_precision")


def rule_scores(score_of):
    """The AP of a fixed rule (it learns nothing) on each of the same 5 folds."""
    return np.array([average_precision_score(train[TARGET].iloc[rows], score_of(train.iloc[rows]))
                     for _, rows in cv.split(train, train[TARGET])])

The cell prints nothing.

## 2. Worked example: the first rows of the log

Score the two baselines and logistic regression on the same folds. "One score for every ticket" gives each ticket 0.117, the train rate. The rule gives 1 to a priority 1 ticket and 0 to the others.

> **Predict.** How far above the floor of 0.117 is the rule? And logistic regression? Write your guesses, then run the cell. It takes a few seconds.

In [ ]:
one_score = rule_scores(lambda tickets: [0.117] * len(tickets))
priority_rule = rule_scores(lambda tickets: (tickets["priority"] == 1).astype(int))
logistic = cv_scores(build_pipeline())
for name, folds in [("one score for all", one_score), ("rule: priority 1", priority_rule),
                    ("logistic regression", logistic)]:
    print(f"{name:20s} folds {folds.round(3)}  CV AP {folds.mean():.3f} ± {folds.std():.3f}")

> **Check.** You should see `CV AP 0.117 ± 0.000`, `CV AP 0.178 ± 0.008` and `CV AP 0.327 ± 0.012`.

Logistic regression is 0.149 above the rule, more than ten times its standard deviation: a real difference. Now confirm the order once on the validation month. Run the cell.

In [ ]:
pipeline = build_pipeline().fit(train[FEATURES], train[TARGET])
print("one score for all  ", round(average_precision_score(valid[TARGET], [0.117] * len(valid)), 3))
print("rule: priority 1   ", round(average_precision_score(valid[TARGET], (valid["priority"] == 1).astype(int)), 3))
print("logistic regression", round(average_precision_score(valid[TARGET], pipeline.predict_proba(valid[FEATURES])[:, 1]), 3))

> **Check.** You should see `0.155`, `0.228` and `0.353`.

The May numbers are higher because 15.5% of the May tickets escalated (11.7% in train). The order is the same.

## 3. The experiment log

The **experiment log** has one row per experiment: `id`, `date`, `change`, `cv_ap`, `cv_std`, `valid_ap` (only for a candidate that you confirm once), `decision` (`baseline`, `keep` or `reject`) and `reason`. It is your memory, it keeps the rejected tries visible, and it explains the final choice to others.

Run the cell. It writes the first three rows and shows the log.

In [ ]:
log_experiment("E01", "Baseline: one score for every ticket (the train rate)", one_score.mean(), one_score.std(),
               0.155, "baseline", "The floor: any useful model must be above it.")
log_experiment("E02", "Baseline: the rule 'flag priority 1', as a 0/1 score", priority_rule.mean(), priority_rule.std(),
               0.228, "baseline", "What Larkfield does today: the model must beat it.")
log_experiment("E03", "Logistic regression, the 10 C04 features", logistic.mean(), logistic.std(),
               0.353, "keep", "Far above both baselines; train AP 0.328, so it does not overfit.")
show_log()

> **Check.** You should see three rows: E01 and E02 with the decision `baseline`, and E03 with `keep`.

## 4. One round of the loop

Tomás suggests `class_weight="balanced"`: in training, each escalated ticket counts about 7.5 times as much as another ticket. It is one change to the best model so far, E03.

> **Predict.** Will the CV AP go up, go down, or stay about the same? Run the cell.

In [ ]:
balanced = cv_scores(build_pipeline(LogisticRegression(max_iter=1000, class_weight="balanced")))
print("logistic regression", logistic.round(3), f"CV AP {logistic.mean():.3f} ± {logistic.std():.3f}")
print("balanced           ", balanced.round(3), f"CV AP {balanced.mean():.3f} ± {balanced.std():.3f}")
print("balanced − logistic", (balanced - logistic).round(3))

> **Check.** You should see `CV AP 0.323 ± 0.012` for the balanced model. The last line shows a negative number on every fold.

The balanced model is lower on all 5 folds. AP depends only on the order of the tickets by score, and the weights change that order very little. Decision: reject.

## 5. Guided practice: log Tomás's experiment

> **Your turn.** Replace `...` with one call of `log_experiment()` for the row E04. Use the change `"E03 with class_weight='balanced'"`, `balanced.mean()` and `balanced.std()`, no valid AP (`None`), the decision `"reject"`, and a reason in your own words. Run the cell, then the check cell.

In [ ]:
# Your turn: write the row E04 for the balanced model, then show the log.
# log_experiment(id, change, cv_ap, cv_std, valid_ap, decision, reason)
...
show_log()

In [ ]:
log = pd.read_csv(LOG_FILE).set_index("id")
if "E04" not in log.index:
    print("Not yet: the log has no row E04. Write it with log_experiment(), then run this cell again.")
else:
    row = log.loc["E04"]
    expect_hash("the row E04", [row["cv_ap"], row["cv_std"], None if pd.isna(row["valid_ap"]) else row["valid_ap"],
                                row["decision"]], "6325e445865d8d5c")

> **Check.** You should see four rows, and `Check passed: the row E04 is right.` The check does not read your reason: any clear sentence is fine.

## 6. Your turn: a wider rule

Change one condition: the baseline. Grace asks about a wider rule: "flag priority 1, **or** any ticket of the payment team" (the team with the highest escalation rate in train, 20.7%).

> **Predict.** Will it beat the rule "flag priority 1" (0.178)?

> **Your turn.** Replace `...` with `rule_scores(lambda tickets: ...)`. The rule is `(tickets["priority"] == 1) | (tickets["team"] == "payment")`, as 0 or 1 (`.astype(int)`). Run the cell, then the check cell.

In [ ]:
# Your turn: the rule "flag priority 1 or team payment" on the same 5 folds.
wider_rule = ...
if wider_rule is not ...:
    print(wider_rule.round(3), f"CV AP {wider_rule.mean():.3f} ± {wider_rule.std():.3f}")

In [ ]:
if wider_rule is ...:
    print("Not yet: replace ... with rule_scores(...), then run the cell again.")
else:
    expect_hash("the CV AP of the wider rule", round(float(wider_rule.mean()), 3), "141693c9211534b1")

> **Your turn.** Now write the row E05: your decision and your reason. Run the cell, then the check cell.

In [ ]:
# Your turn: write the row E05 for the wider rule (your decision and your reason).
...
show_log()

In [ ]:
log = pd.read_csv(LOG_FILE).set_index("id")
if "E05" not in log.index:
    print("Not yet: the log has no row E05. Write it with log_experiment(), then run this cell again.")
else:
    row = log.loc["E05"]
    expect_hash("the row E05", [row["cv_ap"], row["cv_std"], None if pd.isna(row["valid_ap"]) else row["valid_ap"],
                                row["decision"]], "1576c0eaaa540efc")

Compare your decision with the answer on the [lesson page](https://learning.nextia-ai.com/courses/ml/m05/machine-learning-is-iterative/#your-turn-a-wider-rule).

## 7. Failure case: a gain that came from the folds

Tomás tried the balanced model first, but he checked it on another fold than logistic regression. Run the cell.

In [ ]:
print("Tomás's check of logistic regression, fold 2:", logistic[1].round(3))
print("Tomás's check of the balanced model, fold 3: ", balanced[2].round(3))

> **Check.** You should see `0.31` and `0.339`: a "gain" of 0.029.

Fold 3 is easier than fold 2 for any model: logistic regression itself gets 0.342 on fold 3. Compare on the same folds. Run the cell.

In [ ]:
print("fold by fold, balanced − logistic:", (balanced - logistic).round(3))

> **Check.** You should see five negative numbers.

On the same tickets, the balanced model loses every time. **Score every experiment with the same `cv` object and the same metric.** The [lesson page](https://learning.nextia-ai.com/courses/ml/m05/machine-learning-is-iterative/#a-gain-that-came-from-the-folds) has the full failure case.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/ml/m05/machine-learning-is-iterative/#knowledge-check). They count toward your certificate when you are signed in and enrolled.

In this lesson you learned the loop of model development and started the experiment log with five rows. Logistic regression (E03, 0.327 ± 0.012) is the best so far. The next lessons add new model families to the same log.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Decision trees](https://learning.nextia-ai.com/courses/ml/m05/decision-trees/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m05/machine-learning-is-iterative/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.